## Random Forest

In [45]:
pip install fastparquet

Note: you may need to restart the kernel to use updated packages.


In [46]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, classification_report
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

In [23]:
import pandas as pd
from pathlib import Path

root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

panel = pd.read_parquet(
    root / "data/processed_data/analysis_panel.parquet", engine='fastparquet'
)
print(panel.shape)
print(panel.dtypes)
panel.head(20)

(16576, 34)
YEAR                              int16
GEOGRAPHY_CODE                 category
GEOGRAPHY_NAME                 category
IS8_SECTOR                     category
EMPLOYEES                       float32
BUSINESSES                      float32
unemployment_rate               float64
transport_to_employer           float64
drive_to_employer               float64
cycle_to_employer               float64
broadband                       float64
coverage_4g                     float64
gcse_age19                      float64
apprenticeship_starts           float64
apprenticeship_achievements     float64
nvq_level3                      float64
fe_participation                float64
housing_net_additions           float64
enterprise_birth_rate           float64
enterprise_death_rate           float64
enterprise_high_growth_rate     float64
REGION                           object
COUNTY_UA                        object
lq_bus                          float64
emp_share                   

,YEAR,GEOGRAPHY_CODE,GEOGRAPHY_NAME,IS8_SECTOR,EMPLOYEES,BUSINESSES,unemployment_rate,transport_to_employer,drive_to_employer,cycle_to_employer,...,emp_share,lq_emp,gd_emp,gd_bus,n_years_emp,n_years_bus,related_variety,within_sector_diversity,size_large_share,size_micro_share
0,2016,E06000001,Hartlepool,Advanced Manufacturing,1470.0,25.0,4.6,12.900208,7.790802,10.070582,...,0.048197,1.648198,0.001431,-0.112731,8,8,1.032564,1.332179,0.000000,1.000000
1,2016,E06000001,Hartlepool,Creative Industries,450.0,110.0,4.6,12.900208,7.790802,10.070582,...,0.014754,0.281144,0.044382,-0.014263,8,8,1.032564,2.200516,0.000000,1.000000
2,2016,E06000001,Hartlepool,Defence,0.0,0.0,4.6,12.900208,7.790802,10.070582,...,0.000000,0.000000,NaN,NaN,0,0,1.032564,0.000000,NaN,NaN
3,2016,E06000001,Hartlepool,Digital and Technologies,1510.0,440.0,4.6,12.900208,7.790802,10.070582,...,0.049508,0.682440,-0.026657,-0.059004,8,8,1.032564,1.072313,0.000000,1.000000
4,2016,E06000001,Hartlepool,Financial Services,285.0,40.0,4.6,12.900208,7.790802,10.070582,...,0.009344,0.154102,-0.037470,0.047092,8,8,1.032564,1.213008,0.000000,0.750000
5,2016,E06000001,Hartlepool,Life Sciences,40.0,0.0,4.6,12.900208,7.790802,10.070582,...,0.001311,0.473936,-0.073609,NaN,8,0,1.032564,0.000000,NaN,NaN
6,2016,E06000001,Hartlepool,Professional and Business Services,2630.0,870.0,4.6,12.900208,7.790802,10.070582,...,0.086230,0.461236,-0.005547,-0.041919,8,8,1.032564,1.830641,0.000000,0.965517
7,2016,E06000002,Middlesbrough,Advanced Manufacturing,620.0,25.0,5.1,15.372879,8.457817,11.335077,...,0.010622,0.363240,0.005820,0.054637,8,8,1.075517,1.332179,0.000000,0.800000
8,2016,E06000002,Middlesbrough,Creative Industries,1320.0,180.0,5.1,15.372879,8.457817,11.335077,...,0.022614,0.430924,0.055147,0.014442,8,8,1.075517,2.328951,0.000000,1.000000
9,2016,E06000002,Middlesbrough,Defence,0.0,0.0,5.1,15.372879,8.457817,11.335077,...,0.000000,0.000000,NaN,NaN,0,0,1.075517,0.000000,NaN,NaN


In [4]:
panel.info()

<class 'pandas.DataFrame'>
RangeIndex: 16576 entries, 0 to 16575
Data columns (total 34 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   YEAR                         16576 non-null  int16   
 1   GEOGRAPHY_CODE               16576 non-null  category
 2   GEOGRAPHY_NAME               16576 non-null  category
 3   IS8_SECTOR                   16576 non-null  category
 4   EMPLOYEES                    16576 non-null  float32 
 5   BUSINESSES                   16576 non-null  float32 
 6   unemployment_rate            15624 non-null  float64 
 7   transport_to_employer        16520 non-null  float64 
 8   drive_to_employer            16520 non-null  float64 
 9   cycle_to_employer            16520 non-null  float64 
 10  broadband                    16576 non-null  float64 
 11  coverage_4g                  16576 non-null  float64 
 12  gcse_age19                   16520 non-null  float64 
 13  apprenticesh

In [6]:

# ─────────────────────────────────────────────
# 1. CARGAR DATOS
# ─────────────────────────────────────────────
root = next(p for p in Path.cwd().parents if (p / "config.yml").exists())

df = pd.read_parquet(
    root / "data/processed_data/analysis_panel.parquet", engine='fastparquet'
)
print(df.shape)
print(df.dtypes)
df.head(20)

# Filtrar solo 2023
df = df[df["YEAR"] == 2023].copy()
print(f"Shape filtrado (2023): {df.shape}")
print("Sectores IS8:", df["IS8_SECTOR"].unique())

# Dropear missings en cagr_bus
#print(f"Missings en cagr_bus: {df['cagr_bus'].isna().sum()}")
#df = df.dropna(subset=["cagr_bus"])
#print(f"Shape final: {df.shape}")

(16576, 34)
YEAR                              int16
GEOGRAPHY_CODE                 category
GEOGRAPHY_NAME                 category
IS8_SECTOR                     category
EMPLOYEES                       float32
BUSINESSES                      float32
unemployment_rate               float64
transport_to_employer           float64
drive_to_employer               float64
cycle_to_employer               float64
broadband                       float64
coverage_4g                     float64
gcse_age19                      float64
apprenticeship_starts           float64
apprenticeship_achievements     float64
nvq_level3                      float64
fe_participation                float64
housing_net_additions           float64
enterprise_birth_rate           float64
enterprise_death_rate           float64
enterprise_high_growth_rate     float64
REGION                           object
COUNTY_UA                        object
lq_bus                          float64
emp_share                   

In [24]:
# ─────────────────────────────────────────────
# DEFINIR LOS DOS GRUPOS DE FEATURES
# ─────────────────────────────────────────────
features_full = [
   # Entrepreneurial discovery
    "enterprise_birth_rate",
    "enterprise_death_rate",
    "enterprise_high_growth_rate",
    # Connectivity
    "broadband",
    "coverage_4g",
    "transport_to_employer",         # was: public_transport_to_employer
    "drive_to_employer",
    "cycle_to_employer",
    # Human capital
    "nvq_level3",
    "gcse_age19",
    "apprenticeship_starts",
    "apprenticeship_achievements",
    "fe_participation",
    # Labour market
    "unemployment_rate",
    # Place conditions
    "housing_net_additions",
    "size_large_share",
    "size_micro_share",
    # Relatedness
    "related_variety",
    "within_sector_diversity",       # additional EEG variable

]

features_theory = [
     # Human capital
    "nvq_level3",                    # was: level_3+_qualifications
    "gcse_age19",                    # was: gcse_by_age_19
    "apprenticeship_starts",
    "apprenticeship_achievements",
    "fe_participation",              # was: fe_and_skills_participation
    # Entrepreneurial discovery — rates not raw counts
    "enterprise_birth_rate",         # was: new_enterprises (raw)
    "enterprise_death_rate",         # was: deaths_of_enterprises (raw)
    "enterprise_high_growth_rate",   # was: high_growth_enterprises (raw)
    # active_enterprises DROPPED — used only as denominator for rates
    # Connectivity
    "broadband",                     # was: broadband_availability
    "coverage_4g",                   # was: 4g_area_coverage
    # Place conditions
    "housing_net_additions",         # was: net_additions
    # Relatedness — EEG core
    "related_variety",
]

feature_groups = {
    "full":   features_full,
    "theory": features_theory,
}

# ─────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────
RF_PARAMS_REG = dict(n_estimators=100, max_depth=4, min_samples_leaf=5,
                     n_jobs=-1, random_state=42)
RF_PARAMS_CLF = dict(n_estimators=100, max_depth=4, min_samples_leaf=5,
                     n_jobs=-1, random_state=42)

CV_REG = KFold(n_splits=5, shuffle=True, random_state=42)
CV_CLF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dep_vars_regression     = ["lq_emp", "lq_bus", "gd_emp", "gd_bus"]
dep_vars_classification = ["lq_emp", "lq_bus", "gd_emp", "gd_bus"]

sectors = sorted(df["IS8_SECTOR"].dropna().unique())

# ─────────────────────────────────────────────
# LOOP PRINCIPAL — dos grupos
# ─────────────────────────────────────────────
all_results = []
all_fi      = []

for group_name, features in feature_groups.items():

    # Verificar features disponibles
    missing = [f for f in features if f not in df.columns]
    if missing:
        print(f"⚠️  [{group_name}] Columnas no encontradas: {missing}")
    features = [f for f in features if f in df.columns]

    print(f"\n{'#'*60}")
    print(f"  GRUPO: {group_name.upper()}  |  {len(features)} features")
    print(f"{'#'*60}")

    for sector in sectors:
        sub = df[df["IS8_SECTOR"] == sector].copy()
        X   = sub[features].copy()
        X   = X.fillna(X.median(numeric_only=True))

        print(f"\n{'='*60}")
        print(f"  SECTOR: {sector}  |  n = {len(sub)}")
        print(f"{'='*60}")

        # ── REGRESIÓN ────────────────────────────────────────────
        for dep in dep_vars_regression:
            if dep not in sub.columns:
                continue
            y    = sub[dep].copy()
            mask = y.notna()
            X_r, y_r = X[mask], y[mask]

            if len(y_r) < 20:
                print(f"  [SKIP regresión {dep}] n={len(y_r)} < 20")
                continue

            model = RandomForestRegressor(**RF_PARAMS_REG)
            r2_cv   = cross_val_score(model, X_r, y_r, cv=CV_REG, scoring="r2")
            rmse_cv = np.sqrt(-cross_val_score(model, X_r, y_r, cv=CV_REG,
                              scoring="neg_mean_squared_error"))

            print(f"  [Reg {group_name}] {dep} | R²={r2_cv.mean():.3f} ± {r2_cv.std():.3f}")

            # Fit final para feature importance
            model.fit(X_r, y_r)
            all_fi.append(pd.DataFrame({
                "group":      group_name,
                "sector":     sector,
                "dep_var":    dep,
                "model_type": "regression",
                "feature":    features,
                "importance": model.feature_importances_
            }))

            all_results.append({
                "group":        group_name,
                "sector":       sector,
                "dep_var":      dep,
                "model_type":   "regression",
                "n":            int(mask.sum()),
                "R2_mean":      round(r2_cv.mean(), 4),
                "R2_std":       round(r2_cv.std(),  4),
                "RMSE_mean":    round(rmse_cv.mean(), 4),
                "RMSE_std":     round(rmse_cv.std(),  4),
            })

        # ── CLASIFICACIÓN ────────────────────────────────────────
        for dep in dep_vars_classification:
            if dep not in sub.columns:
                continue
            y_cont = sub[dep].copy()
            mask   = y_cont.notna()
            X_c    = X[mask]
            threshold = 1 if dep in ("lq_emp", "lq_bus") else y_cont[mask].median()
            y_c    = (y_cont[mask] >= threshold).astype(int)
            dep_clf = f"{dep}_binary"

            if len(y_c) < 20 or y_c.nunique() < 2:
                print(f"  [SKIP clasificación {dep_clf}] n<20 o clase única")
                continue

            model_c = RandomForestClassifier(**RF_PARAMS_CLF)
            f1_cv  = cross_val_score(model_c, X_c, y_c, cv=CV_CLF, scoring="f1")
            acc_cv = cross_val_score(model_c, X_c, y_c, cv=CV_CLF, scoring="accuracy")

            print(f"  [Clf {group_name}] {dep_clf} | F1={f1_cv.mean():.3f} ± {f1_cv.std():.3f}")

            # Fit final para feature importance
            model_c.fit(X_c, y_c)
            all_fi.append(pd.DataFrame({
                "group":      group_name,
                "sector":     sector,
                "dep_var":    dep_clf,
                "model_type": "classification",
                "feature":    features,
                "importance": model_c.feature_importances_
            }))

            all_results.append({
                "group":      group_name,
                "sector":     sector,
                "dep_var":    dep_clf,
                "model_type": "classification",
                "n":          int(mask.sum()),
                "F1_mean":    round(f1_cv.mean(), 4),
                "F1_std":     round(f1_cv.std(),  4),
                "Acc_mean":   round(acc_cv.mean(), 4),
                "Acc_std":    round(acc_cv.std(),  4),
            })

# ─────────────────────────────────────────────
# EXPORTAR Y COMPARAR
# ─────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
fi_df      = pd.concat(all_fi, ignore_index=True) if all_fi else pd.DataFrame()

# Exportar todo
results_df.to_csv("rf_two_groups_metrics.csv",      index=False)
fi_df.to_csv("rf_two_groups_feature_importance.csv", index=False)

# ── Tabla comparativa regresión ──
reg_df = results_df[results_df["model_type"] == "regression"]
comp_reg = reg_df.pivot_table(
    index=["sector", "dep_var"],
    columns="group",
    values=["R2_mean", "RMSE_mean"]
).round(4)
comp_reg.columns = ["_".join(c) for c in comp_reg.columns]
comp_reg["R2_winner"] = np.where(
    comp_reg["R2_mean_full"] >= comp_reg["R2_mean_theory"], "full", "theory"
)
comp_reg.to_csv("rf_comparison_regression.csv")
print("\n── COMPARACIÓN REGRESIÓN (R²) ──")
print(comp_reg[["R2_mean_full", "R2_mean_theory", "R2_winner"]].to_string())

# ── Tabla comparativa clasificación ──
clf_df = results_df[results_df["model_type"] == "classification"]
comp_clf = clf_df.pivot_table(
    index=["sector", "dep_var"],
    columns="group",
    values=["F1_mean", "Acc_mean"]
).round(4)
comp_clf.columns = ["_".join(c) for c in comp_clf.columns]
comp_clf["F1_winner"] = np.where(
    comp_clf["F1_mean_full"] >= comp_clf["F1_mean_theory"], "full", "theory"
)
comp_clf.to_csv("rf_comparison_classification.csv")
print("\n── COMPARACIÓN CLASIFICACIÓN (F1) ──")
print(comp_clf[["F1_mean_full", "F1_mean_theory", "F1_winner"]].to_string())

print("\n✅ Archivos exportados:")
print("   rf_two_groups_metrics.csv")
print("   rf_two_groups_feature_importance.csv")
print("   rf_comparison_regression.csv")
print("   rf_comparison_classification.csv")


############################################################
  GRUPO: FULL  |  19 features
############################################################

  SECTOR: Advanced Manufacturing  |  n = 296
  [Reg full] lq_emp | R²=0.386 ± 0.131
  [Reg full] lq_bus | R²=0.521 ± 0.059
  [Reg full] gd_emp | R²=-0.040 ± 0.132
  [Reg full] gd_bus | R²=0.062 ± 0.148
  [Clf full] lq_emp_binary | F1=0.716 ± 0.039
  [Clf full] lq_bus_binary | F1=0.792 ± 0.020
  [Clf full] gd_emp_binary | F1=0.488 ± 0.049
  [Clf full] gd_bus_binary | F1=0.572 ± 0.080

  SECTOR: Creative Industries  |  n = 296
  [Reg full] lq_emp | R²=0.634 ± 0.086
  [Reg full] lq_bus | R²=0.757 ± 0.039
  [Reg full] gd_emp | R²=0.052 ± 0.062
  [Reg full] gd_bus | R²=0.108 ± 0.110
  [Clf full] lq_emp_binary | F1=0.591 ± 0.189
  [Clf full] lq_bus_binary | F1=0.810 ± 0.060
  [Clf full] gd_emp_binary | F1=0.599 ± 0.064
  [Clf full] gd_bus_binary | F1=0.593 ± 0.069

  SECTOR: Defence  |  n = 296
  [Reg full] lq_emp | R²=-3.021 ± 4.002
  [Reg

In [35]:
# ─────────────────────────────────────────────
# 5. EXPORTAR
# ─────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
fi_df      = pd.concat(all_fi, ignore_index=True) if all_fi else pd.DataFrame()

results_df.to_csv("rf_metrics.csv",          index=False)
fi_df.to_csv("rf_feature_importance.csv",    index=False)
print("\n✅ Archivos exportados.")

# ─────────────────────────────────────────────
# 6. RESUMEN — separado por tipo de modelo
# ─────────────────────────────────────────────
reg_df = results_df[results_df["model_type"] == "regression"][
    ["sector", "dep_var", "n", "R2_mean", "R2_std", "RMSE_mean", "RMSE_std"]
].copy()

clf_df = results_df[results_df["model_type"] == "classification"][
    ["sector", "dep_var", "n", "F1_mean", "F1_std", "Acc_mean", "Acc_std"]
].copy()

print("\n── REGRESIÓN (RF) ──")
print(reg_df.to_string(index=False))
print("\n── CLASIFICACIÓN (RF) ──")
print(clf_df.to_string(index=False))

reg_df.to_csv("rf_regression_summary.csv",     index=False)
clf_df.to_csv("rf_classification_summary.csv", index=False)
print("\n✅ Tablas de comparación exportadas:")
print("   rf_regression_summary.csv")
print("   rf_classification_summary.csv")


✅ Archivos exportados.

── REGRESIÓN (RF) ──
                            sector dep_var   n  R2_mean  R2_std  RMSE_mean  RMSE_std
            Advanced Manufacturing  lq_emp 296   0.3858  0.1305     0.9572    0.2579
            Advanced Manufacturing  lq_bus 296   0.5214  0.0586     0.2977    0.0159
            Advanced Manufacturing  gd_emp 295  -0.0400  0.1322     0.0516    0.0066
            Advanced Manufacturing  gd_bus 295   0.0619  0.1482     0.0326    0.0060
               Creative Industries  lq_emp 296   0.6341  0.0856     0.3858    0.0627
               Creative Industries  lq_bus 296   0.7570  0.0387     0.2366    0.0218
               Creative Industries  gd_emp 296   0.0516  0.0623     0.0341    0.0039
               Creative Industries  gd_bus 295   0.1079  0.1102     0.0157    0.0022
                           Defence  lq_emp 296  -3.0207  4.0023     6.5715    4.6294
                           Defence  lq_bus 296  -0.2215  0.3922     0.7454    0.5411
                   

In [36]:
# ─────────────────────────────────────────────
# FEATURE IMPORTANCE — formato pivotado por sector
# Una hoja por sector, columnas = dep_vars
# ─────────────────────────────────────────────
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import pandas as pd

TOP_N = 5

dep_vars_reg = ["lq_emp", "lq_bus", "gd_emp", "gd_bus"]
dep_vars_clf = ["lq_emp_binary", "lq_bus_binary", "gd_emp_binary", "gd_bus_binary"]

def build_pivot_for_sector(group_fi, sector, dep_reg, dep_clf, top_n=5):
    """
    Construye la tabla pivotada para un sector:
    filas = top features (union de todas las dep_vars)
    columnas = dep_vars
    separadas en bloque REGRESSION y CLASSIFICATION
    """
    rows = []

    for block_label, dep_list in [("REGRESSION", dep_reg), ("CLASSIFICATION", dep_clf)]:
        # Filtrar sector y dep_vars del bloque
        block_fi = group_fi[
            (group_fi["sector"] == sector) &
            (group_fi["dep_var"].isin(dep_list))
        ]
        if block_fi.empty:
            continue

        # Top N features por dep_var
        top_features = (
            block_fi
            .sort_values("importance", ascending=False)
            .groupby("dep_var")
            .head(top_n)
        )

        # Pivot: filas=feature, columnas=dep_var, valores=importance
        pivot = top_features.pivot_table(
            index="feature",
            columns="dep_var",
            values="importance",
            aggfunc="first"
        ).reindex(columns=dep_list)  # orden fijo de columnas

        # Eliminar filas donde todos son NaN
        pivot = pivot.dropna(how="all")
        pivot = pivot.reset_index()
        pivot.columns.name = None

        # Agregar fila de encabezado del bloque
        rows.append({"block": block_label, "data": pivot})

    return rows

for group_name in ["full", "theory"]:
    group_fi = fi_df[fi_df["group"] == group_name].copy()

    with pd.ExcelWriter(f"feature_importance_{group_name}.xlsx", engine="openpyxl") as writer:

        for sector in sorted(group_fi["sector"].unique()):
            blocks = build_pivot_for_sector(
                group_fi, sector, dep_vars_reg, dep_vars_clf, TOP_N
            )
            if not blocks:
                continue

            # ── Escribir hoja ──
            sheet_name = sector[:31]
            workbook   = writer.book
            worksheet  = workbook.create_sheet(title=sheet_name)

            # Estilos
            header_font    = Font(bold=True, color="FFFFFF")
            header_fill    = PatternFill("solid", fgColor="2F5496")  # azul oscuro
            section_fill   = PatternFill("solid", fgColor="D9E1F2")  # azul claro
            center_align   = Alignment(horizontal="center")

            # Título del sector
            worksheet.cell(row=1, column=1, value=f"Sector: {sector}").font = Font(bold=True, size=12)

            current_row = 2

            for block in blocks:
                label = block["block"]
                pivot = block["data"]
                cols  = pivot.columns.tolist()  # ["feature"] + dep_vars

                # ── Fila etiqueta bloque (REGRESSION / CLASSIFICATION) ──
                cell = worksheet.cell(row=current_row, column=1, value=label)
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = header_fill
                cell.alignment = center_align
                worksheet.merge_cells(
                    start_row=current_row, start_column=1,
                    end_row=current_row,   end_column=len(cols)
                )
                current_row += 1

                # ── Fila de encabezados de columnas ──
                for col_idx, col_name in enumerate(cols, start=1):
                    cell = worksheet.cell(row=current_row, column=col_idx, value=col_name)
                    cell.font = Font(bold=True)
                    cell.fill = section_fill
                    cell.alignment = center_align
                current_row += 1

                # ── Filas de datos ──
                for _, data_row in pivot.iterrows():
                    for col_idx, col_name in enumerate(cols, start=1):
                        val = data_row[col_name]
                        # Redondear importancias, dejar NaN como vacío
                        if col_name != "feature" and pd.notna(val):
                            val = round(float(val), 4)
                        elif pd.isna(val):
                            val = None
                        worksheet.cell(row=current_row, column=col_idx, value=val)
                    current_row += 1

                current_row += 1  # fila vacía entre bloques

            # ── Ajustar ancho de columnas ──
            for col_idx in range(1, len(dep_vars_reg) + 2):
                col_letter = get_column_letter(col_idx)
                worksheet.column_dimensions[col_letter].width = 28 if col_idx == 1 else 14

    print(f"✅ Exportado: feature_importance_{group_name}.xlsx")


✅ Exportado: feature_importance_full.xlsx
✅ Exportado: feature_importance_theory.xlsx


In [37]:
print(fi_df.columns.tolist())
print(fi_df.shape)
print(fi_df.head())

['group', 'sector', 'dep_var', 'model_type', 'feature', 'importance']
(1674, 6)
  group                  sector dep_var  model_type  \
0  full  Advanced Manufacturing  lq_emp  regression   
1  full  Advanced Manufacturing  lq_emp  regression   
2  full  Advanced Manufacturing  lq_emp  regression   
3  full  Advanced Manufacturing  lq_emp  regression   
4  full  Advanced Manufacturing  lq_emp  regression   

                       feature  importance  
0        enterprise_birth_rate    0.007859  
1        enterprise_death_rate    0.003071  
2  enterprise_high_growth_rate    0.003649  
3                    broadband    0.003990  
4                  coverage_4g    0.003730  


In [38]:
print(fi_df.columns.tolist())

['group', 'sector', 'dep_var', 'model_type', 'feature', 'importance']


In [40]:
# summary for lq_emp regression

print("\n── TOP 5 FEATURES por sector (regresión lq_emp) ──────────")
top_fi_lqemp = (
    fi_df[(fi_df["dep_var"] == "lq_emp") & (fi_df["model_type"] == "regression")]
    .sort_values(["sector", "importance"], ascending=[True, False])
    .groupby("sector")
    .head(5)[["sector", "feature", "importance"]]
    .sort_values(["sector", "importance"], ascending=[True, False])
)
print(top_fi_lqemp.to_string(index=False))


── TOP 5 FEATURES por sector (regresión lq_emp) ──────────
                            sector                     feature  importance
            Advanced Manufacturing            size_large_share    0.673170
            Advanced Manufacturing apprenticeship_achievements    0.255312
            Advanced Manufacturing       housing_net_additions    0.200893
            Advanced Manufacturing       apprenticeship_starts    0.146387
            Advanced Manufacturing apprenticeship_achievements    0.097874
               Creative Industries       transport_to_employer    0.324773
               Creative Industries apprenticeship_achievements    0.321999
               Creative Industries            size_large_share    0.295908
               Creative Industries       apprenticeship_starts    0.257758
               Creative Industries                  nvq_level3    0.195257
                           Defence             related_variety    0.454971
                           Defence      

# For growth

# Export Comparative tables

In [41]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Configuración ──────────────────────────────────────────────
OUTPUT_FILE = "feature_importance_por_sector_full_data.xlsx"

DEP_VARS  = ["lq_emp", "lq_bus", "gd_emp", "gd_bus"]
MOD_TYPES = ["regression", "classification"]
TOP_N     = 5

# Renombrar cagr -> gd en dep_var (por si acaso no se hizo antes)
fi_df["dep_var"] = fi_df["dep_var"].str.replace("cagr", "gd")

# Colores por model_type
COLORS = {
    "regression":     {"header": "1F4E79", "subheader": "BDD7EE", "alt": "EBF3FB"},
    "classification": {"header": "375623", "subheader": "C6EFCE", "alt": "EBF5EB"},
}

thin  = Side(style="thin",   color="CCCCCC")
thick = Side(style="medium", color="888888")
BORDER     = Border(left=thin,  right=thin,  top=thin,  bottom=thin)
BORDER_BOT = Border(left=thin,  right=thin,  top=thin,  bottom=thick)


# ── Helpers ────────────────────────────────────────────────────
def style_cell(cell, bold=False, bg=None, font_color="000000",
               align="center", wrap=False, border=BORDER, size=10):
    if bg:
        cell.fill = PatternFill("solid", fgColor=bg)
    cell.font      = Font(bold=bold, color=font_color, name="Calibri", size=size)
    cell.alignment = Alignment(horizontal=align, vertical="center", wrap_text=wrap)
    cell.border    = border


def build_pivot(fi_df, model_type, sector):
    mask = (
        (fi_df["model_type"] == model_type) &
        (fi_df["sector"]     == sector)
    )
    sub = fi_df[mask].copy()

    top = (
        sub.sort_values("importance", ascending=False)
           .groupby("dep_var")
           .head(TOP_N)
    )

    if top.empty:
        return pd.DataFrame(columns=DEP_VARS)

    pivot = (
        top.pivot_table(
            index="feature",
            columns="dep_var",
            values="importance",
            aggfunc="first"
        )
        .reindex(columns=[d for d in DEP_VARS if d in top["dep_var"].unique()])
    )

    pivot["_max"] = pivot.max(axis=1)
    pivot = pivot.sort_values("_max", ascending=False).drop(columns="_max")

    return pivot


# ── Escritura de una tabla (regression o classification) ───────
def write_table(ws, pivot, model_type, start_row, start_col=1):
    c  = COLORS[model_type]
    hc, shc, alt = c["header"], c["subheader"], c["alt"]

    dep_vars_present = list(pivot.columns)
    n_cols = 1 + len(dep_vars_present)

    end_col = start_col + n_cols - 1
    ws.merge_cells(
        start_row=start_row, start_column=start_col,
        end_row=start_row,   end_column=end_col
    )
    title = ws.cell(start_row, start_col, model_type.upper())
    style_cell(title, bold=True, bg=hc, font_color="FFFFFF",
               align="center", size=11)
    ws.row_dimensions[start_row].height = 20

    h_row = start_row + 1
    feat_h = ws.cell(h_row, start_col, "Feature")
    style_cell(feat_h, bold=True, bg=shc, align="left")
    ws.row_dimensions[h_row].height = 16

    for j, dv in enumerate(dep_vars_present):
        dv_h = ws.cell(h_row, start_col + 1 + j, dv)
        style_cell(dv_h, bold=True, bg=shc, align="center")

    for r_idx, (feat, row) in enumerate(pivot.iterrows()):
        data_row = h_row + 1 + r_idx
        bg = alt if r_idx % 2 == 0 else "FFFFFF"
        ws.row_dimensions[data_row].height = 15

        f_cell = ws.cell(data_row, start_col, feat)
        style_cell(f_cell, bg=bg, align="left")

        for j, dv in enumerate(dep_vars_present):
            val = row.get(dv)
            cell = ws.cell(data_row, start_col + 1 + j)
            if pd.notna(val):
                cell.value = round(float(val), 4)
                cell.number_format = "0.0000"
                style_cell(cell, bg=bg, align="center")
            else:
                cell.value = ""
                style_cell(cell, bg="F0F0F0", align="center")

    last_row = h_row + len(pivot)
    return last_row


# ── Exportación principal ──────────────────────────────────────
def export_feature_importance(fi_df, output_file=OUTPUT_FILE):
    sectors = sorted(fi_df["sector"].unique())

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for sector in sectors:
            sheet_name = str(sector)[:31]
            ws = writer.book.create_sheet(title=sheet_name)
            ws.sheet_view.showGridLines = False

            ws.merge_cells("A1:F1")
            t = ws["A1"]
            t.value = f"Sector: {sector}"
            style_cell(t, bold=True, bg="2E2E2E", font_color="FFFFFF",
                       align="center", size=13)
            ws.row_dimensions[1].height = 26

            current_row = 3

            for mt in MOD_TYPES:
                pivot = build_pivot(fi_df, mt, sector)

                if pivot.empty:
                    ws.cell(current_row, 1,
                            f"Sin datos para {mt} en este sector")
                    current_row += 2
                    continue

                last = write_table(ws, pivot, mt,
                                   start_row=current_row, start_col=1)
                current_row = last + 3

            ws.column_dimensions["A"].width = 30
            for j in range(len(DEP_VARS)):
                ws.column_dimensions[get_column_letter(2 + j)].width = 14

        if "Sheet" in writer.book.sheetnames:
            del writer.book["Sheet"]

    print(f"✅ Exportado: {output_file}  ({len(sectors)} sectores)")


# ── Llamada ────────────────────────────────────────────────────
export_feature_importance(fi_df)

✅ Exportado: feature_importance_por_sector_full_data.xlsx  (7 sectores)


In [42]:
reg  = pd.read_csv("rf_comparison_regression.csv")
clf  = pd.read_csv("rf_comparison_classification.csv")

print("── REGRESIÓN ──")
print(reg[["sector","dep_var","R2_mean_full","R2_mean_theory","R2_winner"]].to_string(index=False))

print("\n── CLASIFICACIÓN ──")
print(clf[["sector","dep_var","F1_mean_full","F1_mean_theory","F1_winner"]].to_string(index=False))

── REGRESIÓN ──
                            sector dep_var  R2_mean_full  R2_mean_theory R2_winner
            Advanced Manufacturing  gd_bus        0.0619          0.0301      full
            Advanced Manufacturing  gd_emp       -0.0400         -0.0505      full
            Advanced Manufacturing  lq_bus        0.5214          0.4340      full
            Advanced Manufacturing  lq_emp        0.3858          0.0682      full
               Creative Industries  gd_bus        0.1079          0.0426      full
               Creative Industries  gd_emp        0.0516          0.0482      full
               Creative Industries  lq_bus        0.7570          0.6877      full
               Creative Industries  lq_emp        0.6341          0.4745      full
                           Defence  gd_emp       -0.5431         -0.6064      full
                           Defence  lq_bus       -0.2215         -0.2600      full
                           Defence  lq_emp       -3.0207         -3.048

In [43]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# DATOS
# ─────────────────────────────────────────────
reg = pd.read_csv("rf_comparison_regression.csv")
clf = pd.read_csv("rf_comparison_classification.csv")

# ─────────────────────────────────────────────
# UMBRALES Y COLORES
# ─────────────────────────────────────────────
# Regresión R²
def r2_fill(val):
    if pd.isna(val):
        return PatternFill("solid", fgColor="D9D9D9")  # gris
    if val >= 0.30:
        return PatternFill("solid", fgColor="C6EFCE")  # verde
    if val >= 0.10:
        return PatternFill("solid", fgColor="FFEB9C")  # amarillo
    return PatternFill("solid", fgColor="FFC7CE")      # rojo

# Clasificación F1
def f1_fill(val):
    if pd.isna(val):
        return PatternFill("solid", fgColor="D9D9D9")
    if val >= 0.65:
        return PatternFill("solid", fgColor="C6EFCE")  # verde
    if val >= 0.50:
        return PatternFill("solid", fgColor="FFEB9C")  # amarillo
    return PatternFill("solid", fgColor="FFC7CE")      # rojo

# Estilos generales
bold          = Font(bold=True)
white_bold    = Font(bold=True, color="FFFFFF")
header_fill   = PatternFill("solid", fgColor="2F5496")
section_fill  = PatternFill("solid", fgColor="D9E1F2")
center        = Alignment(horizontal="center", vertical="center", wrap_text=True)
left          = Alignment(horizontal="left",   vertical="center")
thin_border   = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"),  bottom=Side(style="thin")
)

# ─────────────────────────────────────────────
# HELPER: escribir celda con estilo
# ─────────────────────────────────────────────
def write_cell(ws, row, col, value, font=None, fill=None, alignment=None, border=None):
    cell = ws.cell(row=row, column=col, value=value)
    if font:      cell.font      = font
    if fill:      cell.fill      = fill
    if alignment: cell.alignment = alignment
    if border:    cell.border    = border
    return cell

# ─────────────────────────────────────────────
# CONSTRUIR EXCEL
# ─────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)  # quitar hoja vacía default

sectors = sorted(reg["sector"].unique())

for sector in sectors:
    ws = wb.create_sheet(title=sector[:31])

    reg_s = reg[reg["sector"] == sector].copy()
    clf_s = clf[clf["sector"] == sector].copy()

    # ── TÍTULO ──────────────────────────────
    ws.merge_cells("A1:E1")
    write_cell(ws, 1, 1, f"Sector: {sector}",
               font=Font(bold=True, size=13, color="FFFFFF"),
               fill=PatternFill("solid", fgColor="1F3864"),
               alignment=center)
    ws.row_dimensions[1].height = 22

    # ── LEYENDA ─────────────────────────────
    ws.merge_cells("A2:E2")
    write_cell(ws, 2, 1,
               "🟢 Bueno (R²≥0.30 / F1≥0.65)   🟡 Marginal (R²0.10–0.29 / F1 0.50–0.64)   🔴 No recomendado",
               font=Font(italic=True, size=9),
               alignment=center)
    ws.row_dimensions[2].height = 18

    current_row = 3

    # ════════════════════════════════════════
    # BLOQUE REGRESIÓN
    # ════════════════════════════════════════
    ws.merge_cells(f"A{current_row}:E{current_row}")
    write_cell(ws, current_row, 1, "REGRESIÓN — R² (CV medio)",
               font=white_bold, fill=header_fill, alignment=center)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    # Encabezados regresión
    reg_headers = ["Variable dependiente", "R² Full", "R² Theory", "Mejor grupo", "Valoración"]
    for col_idx, h in enumerate(reg_headers, start=1):
        write_cell(ws, current_row, col_idx, h,
                   font=bold, fill=section_fill,
                   alignment=center, border=thin_border)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    # Filas regresión
    for _, row_data in reg_s.iterrows():
        dep      = row_data["dep_var"]
        r2_full  = row_data["R2_mean_full"]
        r2_theory= row_data["R2_mean_theory"]
        winner   = row_data["R2_winner"]
        best_r2  = max(r2_full, r2_theory)

        # Valoración textual
        if best_r2 >= 0.30:
            valoracion = "✅ Recomendado"
        elif best_r2 >= 0.10:
            valoracion = "⚠️ Marginal"
        else:
            valoracion = "❌ No usar"

        write_cell(ws, current_row, 1, dep,        alignment=left,   border=thin_border)
        write_cell(ws, current_row, 2, round(r2_full,   4) if pd.notna(r2_full)   else "—",
                   fill=r2_fill(r2_full),   alignment=center, border=thin_border)
        write_cell(ws, current_row, 3, round(r2_theory, 4) if pd.notna(r2_theory) else "—",
                   fill=r2_fill(r2_theory), alignment=center, border=thin_border)
        write_cell(ws, current_row, 4, winner,     alignment=center, border=thin_border)
        write_cell(ws, current_row, 5, valoracion,
                   fill=r2_fill(best_r2),   alignment=center, border=thin_border)
        current_row += 1

    current_row += 1  # espacio

    # ════════════════════════════════════════
    # BLOQUE CLASIFICACIÓN
    # ════════════════════════════════════════
    ws.merge_cells(f"A{current_row}:E{current_row}")
    write_cell(ws, current_row, 1, "CLASIFICACIÓN — F1 (CV medio)",
               font=white_bold, fill=header_fill, alignment=center)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    # Encabezados clasificación
    clf_headers = ["Variable dependiente", "F1 Full", "F1 Theory", "Mejor grupo", "Valoración"]
    for col_idx, h in enumerate(clf_headers, start=1):
        write_cell(ws, current_row, col_idx, h,
                   font=bold, fill=section_fill,
                   alignment=center, border=thin_border)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    # Filas clasificación
    for _, row_data in clf_s.iterrows():
        dep      = row_data["dep_var"]
        f1_full  = row_data["F1_mean_full"]
        f1_theory= row_data["F1_mean_theory"]
        winner   = row_data["F1_winner"]
        best_f1  = max(f1_full, f1_theory)

        if best_f1 >= 0.65:
            valoracion = "✅ Recomendado"
        elif best_f1 >= 0.50:
            valoracion = "⚠️ Marginal"
        else:
            valoracion = "❌ No usar"

        write_cell(ws, current_row, 1, dep,         alignment=left,   border=thin_border)
        write_cell(ws, current_row, 2, round(f1_full,   4) if pd.notna(f1_full)   else "—",
                   fill=f1_fill(f1_full),   alignment=center, border=thin_border)
        write_cell(ws, current_row, 3, round(f1_theory, 4) if pd.notna(f1_theory) else "—",
                   fill=f1_fill(f1_theory), alignment=center, border=thin_border)
        write_cell(ws, current_row, 4, winner,      alignment=center, border=thin_border)
        write_cell(ws, current_row, 5, valoracion,
                   fill=f1_fill(best_f1),   alignment=center, border=thin_border)
        current_row += 1

    # ── Ajustar anchos ───────────────────────
    col_widths = [32, 12, 12, 14, 18]
    for i, w in enumerate(col_widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w

wb.save("rf_model_quality_summary.xlsx")
print("✅ Exportado: rf_model_quality_summary.xlsx")


✅ Exportado: rf_model_quality_summary.xlsx


In [44]:
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─────────────────────────────────────────────
# CODEBOOK
# ─────────────────────────────────────────────
cb = pd.read_excel("PP422_Variable_Codebook.xlsx")
cb = cb[["variable", "group", "definition"]].dropna(subset=["variable"])
cb["variable"] = cb["variable"].str.strip()
codebook = cb.set_index("variable").to_dict(orient="index")

def get_meta(feature, field):
    entry = codebook.get(feature.strip(), {})
    return entry.get(field, "—")

# ─────────────────────────────────────────────
# RENOMBRAR cagr -> gd
# ─────────────────────────────────────────────
fi_df["dep_var"] = fi_df["dep_var"].str.replace("cagr", "gd")

# ─────────────────────────────────────────────
# UMBRALES
# ─────────────────────────────────────────────
R2_GOOD     = 0.30
R2_MARGINAL = 0.10
F1_GOOD     = 0.65
F1_MARGINAL = 0.50
CUM_THRESH  = 0.80
MIN_FEAT    = 2
MAX_FEAT    = 5

# ─────────────────────────────────────────────
# MÉTRICAS
# ─────────────────────────────────────────────
reg = pd.read_csv("rf_comparison_regression.csv")
clf = pd.read_csv("rf_comparison_classification.csv")

reg["R2_best"] = reg[["R2_mean_full", "R2_mean_theory"]].max(axis=1)
clf["F1_best"] = clf[["F1_mean_full", "F1_mean_theory"]].max(axis=1)

# ─────────────────────────────────────────────
# FEATURE IMPORTANCE
# ─────────────────────────────────────────────
fi = fi_df[fi_df["group"] == "theory"].copy()

def get_top_features(fi_data, sector, dep_var,
                     cum_thresh=CUM_THRESH, min_f=MIN_FEAT, max_f=MAX_FEAT):
    sub = fi_data[
        (fi_data["sector"]  == sector) &
        (fi_data["dep_var"] == dep_var)
    ].sort_values("importance", ascending=False).reset_index(drop=True)
    if sub.empty:
        return pd.DataFrame()
    sub["cumulative"] = sub["importance"].cumsum()
    n = int((sub["cumulative"] < cum_thresh).sum()) + 1
    n = max(min_f, min(n, max_f))
    return sub[["feature", "importance"]].head(n).reset_index(drop=True)

# ─────────────────────────────────────────────
# ESTILOS
# ─────────────────────────────────────────────
FILLS = {
    "good":    PatternFill("solid", fgColor="C6EFCE"),
    "marginal":PatternFill("solid", fgColor="FFEB9C"),
    "bad":     PatternFill("solid", fgColor="FFC7CE"),
    "header":  PatternFill("solid", fgColor="2F5496"),
    "section": PatternFill("solid", fgColor="D9E1F2"),
    "title":   PatternFill("solid", fgColor="1F3864"),
    "feat_bg": PatternFill("solid", fgColor="F2F2F2"),
}

def quality_fill(val, metric="r2"):
    good, marg = (R2_GOOD, R2_MARGINAL) if metric == "r2" else (F1_GOOD, F1_MARGINAL)
    if pd.isna(val) or val < marg: return FILLS["bad"]
    if val < good:                 return FILLS["marginal"]
    return FILLS["good"]

def quality_label(val, metric="r2"):
    good, marg = (R2_GOOD, R2_MARGINAL) if metric == "r2" else (F1_GOOD, F1_MARGINAL)
    if pd.isna(val) or val < marg: return "❌ Not recommended"
    if val < good:                 return "⚠️ Marginal"
    return "✅ Recommended"

thin   = Side(style="thin")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)
CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT   = Alignment(horizontal="left",   vertical="center", wrap_text=True)
BOLD   = Font(bold=True)
WHITE_BOLD = Font(bold=True, color="FFFFFF")

def wc(ws, row, col, value, font=None, fill=None, align=None):
    c = ws.cell(row=row, column=col, value=value)
    if font:  c.font      = font
    if fill:  c.fill      = fill
    if align: c.alignment = align
    c.border = BORDER
    return c

# ─────────────────────────────────────────────
# COLUMNAS
# ─────────────────────────────────────────────
HEADERS_REG = ["Dependent Variable", "R² (best)", "Quality",
               "Feature", "Importance", "Cumul.", "Group", "Definition"]
HEADERS_CLF = ["Dependent Variable", "F1 (best)", "Quality",
               "Feature", "Importance", "Cumul.", "Group", "Definition"]
NCOLS = 8

REG_DEPS = ["lq_emp", "lq_bus"]
CLF_DEPS = ["lq_emp_binary", "lq_bus_binary",
            "gd_emp_binary", "gd_bus_binary"]

# ─────────────────────────────────────────────
# HELPER: fila de warning
# ─────────────────────────────────────────────
def write_warning_row(ws, current_row, dep, metric_val, fill_q, label, msg):
    wc(ws, current_row, 1, dep,   align=LEFT)
    wc(ws, current_row, 2,
       round(float(metric_val), 4) if pd.notna(metric_val) else "—",
       fill=fill_q, align=CENTER)
    wc(ws, current_row, 3, label, fill=fill_q, align=CENTER)
    wc(ws, current_row, 4, msg,
       font=Font(italic=True, color="9C0006"),
       fill=FILLS["bad"], align=LEFT)
    for col in range(5, NCOLS + 1):
        wc(ws, current_row, col, "", fill=FILLS["bad"])
    ws.merge_cells(
        start_row=current_row, start_column=4,
        end_row=current_row,   end_column=NCOLS
    )
    return current_row + 1

# ─────────────────────────────────────────────
# HELPER: bloque de features
# ─────────────────────────────────────────────
def write_features_block(ws, current_row, dep, metric_val,
                         fill_q, label, top):
    first_row = current_row
    for i, feat_row in top.iterrows():
        feat = feat_row["feature"]
        imp  = feat_row["importance"]
        cum  = top["importance"].iloc[:i+1].sum()
        grp  = get_meta(feat, "group")
        defn = get_meta(feat, "definition")

        if i == 0:
            wc(ws, current_row, 1, dep, align=LEFT)
            wc(ws, current_row, 2, round(float(metric_val), 4),
               fill=fill_q, align=CENTER)
            wc(ws, current_row, 3, label, fill=fill_q, align=CENTER)
        else:
            wc(ws, current_row, 1, "", align=LEFT)
            wc(ws, current_row, 2, "", fill=fill_q, align=CENTER)
            wc(ws, current_row, 3, "", fill=fill_q, align=CENTER)

        wc(ws, current_row, 4, feat,
           fill=FILLS["feat_bg"], align=LEFT)
        wc(ws, current_row, 5, round(float(imp), 4),
           fill=FILLS["feat_bg"], align=CENTER)
        wc(ws, current_row, 6, f"{cum:.1%}",
           fill=FILLS["feat_bg"], align=CENTER)
        wc(ws, current_row, 7, grp,
           fill=FILLS["feat_bg"], align=LEFT)
        wc(ws, current_row, 8, defn,
           fill=FILLS["feat_bg"], align=LEFT)

        current_row += 1

    if len(top) > 1:
        for col in [1, 2, 3]:
            ws.merge_cells(
                start_row=first_row, start_column=col,
                end_row=current_row - 1, end_column=col
            )
            c = ws.cell(first_row, col)
            c.alignment = LEFT if col == 1 else CENTER
            if col in [2, 3]:
                c.fill = fill_q
            c.border = BORDER

    return current_row

# ─────────────────────────────────────────────
# CONSTRUIR EXCEL
# ─────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

sectors = sorted(fi["sector"].unique())

for sector in sectors:
    ws = wb.create_sheet(title=sector[:31])

    # ── Título ──────────────────────────────
    ws.merge_cells(f"A1:{get_column_letter(NCOLS)}1")
    wc(ws, 1, 1, f"Sector: {sector}",
       font=Font(bold=True, size=13, color="FFFFFF"),
       fill=FILLS["title"], align=CENTER)
    ws.row_dimensions[1].height = 24

    # ── Leyenda ─────────────────────────────
    ws.merge_cells(f"A2:{get_column_letter(NCOLS)}2")
    wc(ws, 2, 1,
       "🟢 Good (R²≥0.30 / F1≥0.65)   "
       "🟡 Marginal (R²0.10–0.29 / F1 0.50–0.64)   "
       "🔴 Not recommended  |  "
       "Features: top 80% cumulative importance (min 2, max 5)",
       font=Font(italic=True, size=9), align=CENTER)
    ws.row_dimensions[2].height = 16

    current_row = 3

    # ════════════════════════════════════════
    # BLOQUE REGRESIÓN
    # ════════════════════════════════════════
    ws.merge_cells(f"A{current_row}:{get_column_letter(NCOLS)}{current_row}")
    wc(ws, current_row, 1, "REGRESSION — Location Quotient (LQ)",
       font=WHITE_BOLD, fill=FILLS["header"], align=CENTER)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    # Aviso gd
    ws.merge_cells(f"A{current_row}:{get_column_letter(NCOLS)}{current_row}")
    wc(ws, current_row, 1,
       "⚠️  Growth (gd) not shown in regression "
       "— R² ≈ 0 across all sectors. Use binary classification instead.",
       font=Font(italic=True, size=9, color="9C0006"),
       fill=FILLS["bad"], align=CENTER)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    # Encabezados
    for col, h in enumerate(HEADERS_REG, start=1):
        wc(ws, current_row, col, h,
           font=BOLD, fill=FILLS["section"], align=CENTER)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    for dep in REG_DEPS:
        r2_row = reg[(reg["sector"] == sector) & (reg["dep_var"] == dep)]
        r2_val = r2_row["R2_best"].values[0] if not r2_row.empty else np.nan
        fill_q = quality_fill(r2_val, "r2")
        label  = quality_label(r2_val, "r2")

        if pd.isna(r2_val) or r2_val < R2_MARGINAL:
            current_row = write_warning_row(
                ws, current_row, dep, r2_val, fill_q, label,
                "❌ Insufficient R² — features not shown"
            )
        else:
            top = get_top_features(fi, sector, dep)
            if not top.empty:
                current_row = write_features_block(
                    ws, current_row, dep, r2_val, fill_q, label, top
                )
            else:
                current_row = write_warning_row(
                    ws, current_row, dep, r2_val, fill_q, label,
                    "⚠️ No feature importance data available"
                )
        current_row += 1

    current_row += 1

    # ════════════════════════════════════════
    # BLOQUE CLASIFICACIÓN
    # ════════════════════════════════════════
    ws.merge_cells(f"A{current_row}:{get_column_letter(NCOLS)}{current_row}")
    wc(ws, current_row, 1, "BINARY CLASSIFICATION — LQ and Growth (gd)",
       font=WHITE_BOLD, fill=FILLS["header"], align=CENTER)
    ws.row_dimensions[current_row].height = 18
    current_row += 1

    # Encabezados
    for col, h in enumerate(HEADERS_CLF, start=1):
        wc(ws, current_row, col, h,
           font=BOLD, fill=FILLS["section"], align=CENTER)
    ws.row_dimensions[current_row].height = 16
    current_row += 1

    for dep in CLF_DEPS:
        f1_row = clf[(clf["sector"] == sector) & (clf["dep_var"] == dep)]
        f1_val = f1_row["F1_best"].values[0] if not f1_row.empty else np.nan
        fill_q = quality_fill(f1_val, "f1")
        label  = quality_label(f1_val, "f1")
        dep_fi = dep.replace("_binary", "")

        if pd.isna(f1_val) or f1_val < F1_MARGINAL:
            current_row = write_warning_row(
                ws, current_row, dep, f1_val, fill_q, label,
                "❌ Insufficient F1 — features not shown"
            )
        else:
            top = get_top_features(fi, sector, dep_fi)
            if not top.empty:
                current_row = write_features_block(
                    ws, current_row, dep, f1_val, fill_q, label, top
                )
            else:
                current_row = write_warning_row(
                    ws, current_row, dep, f1_val, fill_q, label,
                    "⚠️ No feature importance data available"
                )
        current_row += 1

    # ── Ajustar anchos ───────────────────────
    for i, w in enumerate([22, 11, 18, 26, 12, 9, 26, 55], start=1):
        ws.column_dimensions[get_column_letter(i)].width = w

    for row in ws.iter_rows():
        for cell in row:
            if cell.column == 8 and cell.value and len(str(cell.value)) > 80:
                ws.row_dimensions[cell.row].height = 45

wb.save("rf_feature_importance_filtered.xlsx")
print("✅ Exported: rf_feature_importance_filtered.xlsx")

✅ Exported: rf_feature_importance_filtered.xlsx
